# Gated Attention for LLMs — Evaluation Notebook
Standalone evaluation of all 7 trained variants (Baseline + G1–G5 + PNG).
Loads checkpoints from v3, runs full analysis, produces all figures.

| Variant | Gate position | Gate equation |
|---------|--------------|---------------|
| Baseline | — | no gate |
| G1 | Post-SDPA | `out *= σ(X·Wθ)` |
| G2 | Value proj | `v *= σ(X·Wθ)` |
| G3 | Key proj | `k *= σ(X·Wθ)` |
| G4 | Query proj | `q *= σ(X·Wθ)` |
| G5 | Post-concat | `out *= σ(X·Wθ)` |
| **PNG** | Post-SDPA (norm-aware) | `out *= σ(X·Wθ − β‖X‖₂e)` |

**No training in this notebook.** All models are loaded from `checkpoints_llm_v3/`.

## 0. Setup & Config

In [ ]:
import os, types, math, json, glob as _glob
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.stats import pearsonr, spearmanr
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from datasets import load_dataset

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── Config ─────────────────────────────────────────────────────────────────
SEQ_LEN      = 512
BATCH_SIZE   = 8
EVAL_BATCHES = None      # None = full test set
SINK_BATCHES = 20
GATE_BATCHES = 20
DISPLAY_LEN  = 40

# ── Checkpoint search — tries multiple locations in priority order ─────────
# Priority 1: v3 final checkpoints (gpt2_<name>.pth)
# Priority 2: v2 final checkpoints (gpt2_<name>.pth)
# Priority 3: epoch checkpoints from Kaggle input (e.g. Baseline_epoch5.pth)
_SEARCH_DIRS = [
    Path('checkpoints_llm_v3'),
    Path('checkpoints_llm_unfrozen'),
    *[Path(p) for p in _glob.glob('/kaggle/input/**/checkpoints_llm_unfrozen', recursive=True)],
    *[Path(p) for p in _glob.glob('/kaggle/input/**/checkpoints_llm_v3',       recursive=True)],
]

FIG_DIR = Path('eval_figures')
FIG_DIR.mkdir(exist_ok=True)

def _safe_name(n):
    return n.replace(' ', '_').replace('—', '').replace('(', '').replace(')', '').replace('/', '')

EXPERIMENTS = [
    ('Baseline',          'baseline'),
    ('G1 — SDPA output',  'G1'),
    ('G2 — Value proj',   'G2'),
    ('G3 — Key proj',     'G3'),
    ('G4 — Query proj',   'G4'),
    ('G5 — Dense output', 'G5'),
    ('PNG — Norm-Aware',  'PNG'),
]

PALETTE = {
    'Baseline':          '#888888',
    'G1 — SDPA output':  '#4C72B0',
    'G2 — Value proj':   '#55A868',
    'G3 — Key proj':     '#C44E52',
    'G4 — Query proj':   '#DD8452',
    'G5 — Dense output': '#937860',
    'PNG — Norm-Aware':  '#9467BD',
}


def find_ckpt(name):
    """
    Search for the best available checkpoint for a given variant.
    Tries (in order):
      1. checkpoints_llm_v3/gpt2_<safe_name>.pth      (v3 final)
      2. checkpoints_llm_unfrozen/gpt2_<safe_name>.pth (v2 final)
      3. Any dir: gpt2_<safe_name>.pth
      4. Any dir: <safe_name>_epoch*.pth  (latest epoch ckpt)
      5. Any dir: <safe_name>.pth
    """
    sname = _safe_name(name)
    # Patterns to try, in priority order
    candidates = []
    for d in _SEARCH_DIRS:
        if not d.exists(): continue
        candidates += list(d.glob(f'gpt2_{sname}.pth'))
    for d in _SEARCH_DIRS:
        if not d.exists(): continue
        # latest epoch checkpoint
        ep_ckpts = sorted(d.glob(f'{sname}_epoch*.pth'))
        if ep_ckpts: candidates.append(ep_ckpts[-1])
        candidates += list(d.glob(f'{sname}.pth'))

    if not candidates:
        return None
    return candidates[0]   # first = highest priority


print(f'torch {torch.__version__} | device={DEVICE}')
print(f'FIG_DIR : {FIG_DIR.resolve()}')
print()
print('Checkpoint inventory:')
for name, pos in EXPERIMENTS:
    ckpt = find_ckpt(name)
    if ckpt:
        d  = torch.load(ckpt, map_location='cpu', weights_only=False)
        ed = d.get('epochs_done', d.get('epoch', '?'))
        sz = ckpt.stat().st_size / 1e6
        print(f'  ✓  {name:<26}  {ckpt.name:<40}  epochs={ed}  ({sz:.0f} MB)')
    else:
        print(f'  ✗  {name:<26}  NOT FOUND')

## 1. Dataset — WikiText-103 (test split)

In [ ]:
tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

raw = load_dataset('wikitext', 'wikitext-103-raw-v1')

def tokenize_and_chunk(split, max_tokens=None):
    ids_list, collected = [], 0
    limit = max_tokens or int(1e18)
    for row in raw[split]:
        text = row['text'].strip()
        if not text: continue
        toks = tokenizer(text, add_special_tokens=False, return_tensors='pt')['input_ids'][0]
        if collected + len(toks) > limit:
            toks = toks[:limit - collected]
        ids_list.append(toks)
        collected += len(toks)
        if collected >= limit: break
    all_ids = torch.cat(ids_list)
    n = len(all_ids) // SEQ_LEN
    return all_ids[:n * SEQ_LEN].reshape(n, SEQ_LEN)

class TokenDataset(Dataset):
    def __init__(self, c): self.c = c
    def __len__(self):     return len(self.c)
    def __getitem__(self, i): return self.c[i]

print('Tokenising val + test ...')
val_ids  = tokenize_and_chunk('validation')
test_ids = tokenize_and_chunk('test')

val_loader  = DataLoader(TokenDataset(val_ids),  BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(TokenDataset(test_ids), BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Val  : {len(val_ids):>5} chunks  ({len(val_ids)*SEQ_LEN/1e6:.2f}M tokens)')
print(f'Test : {len(test_ids):>5} chunks  ({len(test_ids)*SEQ_LEN/1e6:.2f}M tokens)')

## 2. Gate Modules & Model Builder

In [ ]:
from transformers.models.gpt2.modeling_gpt2 import GPT2Attention

def _split_heads(t, nh, hd):
    return t.view(t.size()[:-1] + (nh, hd)).permute(0, 2, 1, 3)

def _merge_heads(t, nh, hd):
    return t.permute(0, 2, 1, 3).contiguous().view(t.size(0), t.size(2), nh * hd)

def _apply_gate(t_bhnd, g_bnh):
    return t_bhnd * g_bnh.permute(0, 2, 1).unsqueeze(-1)


class HeadGate(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.W = nn.Parameter(torch.zeros(d, h))
    def forward(self, x):
        return torch.sigmoid(x @ self.W)


class PNGGate(nn.Module):
    def __init__(self, d, h, beta=0.1):
        super().__init__()
        self.W    = nn.Parameter(torch.zeros(d, h))
        self.e    = nn.Parameter(torch.zeros(h))
        self.beta = beta
    def forward(self, x):
        return torch.sigmoid(x @ self.W - self.beta * x.norm(dim=-1, keepdim=True) * self.e)


def make_gated_forward(attn_module, gate, pos):
    n_head = attn_module.num_heads; head_size = attn_module.head_dim; embed_dim = attn_module.embed_dim
    def forward(self, hidden_states, **kwargs):
        B, N, _ = hidden_states.shape; x = hidden_states; g = gate(x)
        layer_past = kwargs.get('past_key_values', kwargs.get('layer_past', None))
        attention_mask = kwargs.get('attention_mask', None)
        use_cache = kwargs.get('use_cache', False)
        output_attentions = kwargs.get('output_attentions', False)
        qkv = self.c_attn(hidden_states)
        q, k, v = qkv.split(self.embed_dim, dim=2)
        q = _split_heads(q, n_head, head_size); k = _split_heads(k, n_head, head_size); v = _split_heads(v, n_head, head_size)
        if pos == 'G4': q = _apply_gate(q, g)
        if pos == 'G3': k = _apply_gate(k, g)
        if pos == 'G2': v = _apply_gate(v, g)
        if layer_past is not None and isinstance(layer_past, tuple):
            k = torch.cat([layer_past[0], k], dim=-2); v = torch.cat([layer_past[1], v], dim=-2)
        present = (k, v) if use_cache else None
        if output_attentions or getattr(self, '_capture_attn', False):
            scale = head_size ** -0.5; scores = (q * scale) @ k.transpose(-2, -1)
            if attention_mask is not None: scores = scores + attention_mask
            if layer_past is None:
                Nq, Nk = q.size(-2), k.size(-2)
                cm = torch.triu(torch.ones(Nq, Nk, device=x.device, dtype=torch.bool), diagonal=1)
                scores = scores.masked_fill(cm[None, None], float('-inf'))
            attn_w = scores.softmax(-1); attn_w = self.attn_dropout(attn_w); attn_out = attn_w @ v
            if getattr(self, '_capture_attn', False): self._last_attn = attn_w.detach()
        else:
            attn_out = F.scaled_dot_product_attention(q, k, v, attn_mask=attention_mask,
                dropout_p=self.attn_dropout.p if self.training else 0.0, is_causal=(layer_past is None))
            attn_w = None
        if pos in ('G1', 'PNG'):
            attn_out = _apply_gate(attn_out, g)
        # ── Capture gate & norm for ALL positions (Fig 5/6/7) ──
        if getattr(self, '_capture_gate', False):
            self._last_gate   = g.detach()
            self._last_x_norm = x.norm(dim=-1).detach()
        attn_out = _merge_heads(attn_out, n_head, head_size)
        if pos == 'G5':
            attn_out = (attn_out.view(B, N, n_head, head_size) * g.unsqueeze(-1)).view(B, N, embed_dim)
        attn_out = self.c_proj(attn_out); attn_out = self.resid_dropout(attn_out)
        outputs = (attn_out, present)
        if output_attentions: outputs += (attn_w,)
        return outputs
    return types.MethodType(forward, attn_module)


def inject_gates_gpt2(model, pos):
    d, h = model.config.n_embd, model.config.n_head
    gate_list = []
    for block in model.transformer.h:
        g = PNGGate(d, h) if pos == 'PNG' else HeadGate(d, h)
        block.attn.forward = make_gated_forward(block.attn, g, pos)
        gate_list.append(g)
    model.gate_params = nn.ModuleList(gate_list)
    return model


def build_model(pos='baseline'):
    m = GPT2LMHeadModel.from_pretrained('gpt2')
    if pos != 'baseline':
        inject_gates_gpt2(m, pos)
    m = m.to(DEVICE)
    return m


def load_model(name, pos):
    """Load trained checkpoint using find_ckpt(). Always del + empty_cache() after use."""
    ckpt = find_ckpt(name)
    if ckpt is None:
        raise FileNotFoundError(
            f'No checkpoint found for "{name}". '
            f'Make sure checkpoints_llm_unfrozen/ or checkpoints_llm_v3/ is available, '
            f'or attach the previous notebook output as a Kaggle input dataset.'
        )
    m = build_model(pos)
    data = torch.load(ckpt, map_location=DEVICE, weights_only=False)
    m.load_state_dict(data['state'])
    m.eval()
    print(f'  Loaded {ckpt.name}')
    return m


def load_log(name):
    """Load training log (loss/ppl curves) without loading model weights to GPU."""
    ckpt = find_ckpt(name)
    if ckpt is None:
        return {'step': [], 'train_loss': [], 'val_ppl': []}
    data = torch.load(ckpt, map_location='cpu', weights_only=False)
    return data.get('log', {'step': [], 'train_loss': [], 'val_ppl': []})


print('Gate modules, build_model, load_model, load_log — ready.')

## 3. Perplexity Evaluation — All 7 Variants

In [ ]:
@torch.no_grad()
def evaluate_ppl(model, loader, max_batches=EVAL_BATCHES, desc=''):
    model.eval()
    total_loss, total_tokens = 0.0, 0
    for i, batch in enumerate(tqdm(loader, desc=desc, leave=False)):
        if max_batches and i >= max_batches: break
        ids = batch.to(DEVICE); B, N = ids.shape
        out = model(ids, labels=ids)
        total_loss   += out.loss.item() * B * (N - 1)
        total_tokens += B * (N - 1)
    return math.exp(total_loss / total_tokens)


test_ppls = {}
val_ppls  = {}

for name, pos in EXPERIMENTS:
    print(f'Evaluating {name} ...')
    m = load_model(name, pos)
    test_ppls[name] = evaluate_ppl(m, test_loader, desc=f'{name} test')
    val_ppls[name]  = evaluate_ppl(m, val_loader,  desc=f'{name} val')
    del m
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    print(f'  val PPL={val_ppls[name]:.2f}  test PPL={test_ppls[name]:.2f}')

print()
print(f'  {"Variant":<26}  {"Val PPL":>8}  {"Test PPL":>9}  {"ΔPPL (test)":>11}')
print('  ' + '-'*60)
base_ppl = test_ppls['Baseline']
for name, _ in EXPERIMENTS:
    delta = base_ppl - test_ppls[name]
    best  = ' ← best' if test_ppls[name] == min(test_ppls.values()) else ''
    print(f'  {name:<26}  {val_ppls[name]:>8.2f}  {test_ppls[name]:>9.2f}  {delta:>+11.2f}{best}')

## 4. Attention Sink Analysis
Measures how much attention each layer places on the first token — the "sink" phenomenon identified in the paper.

In [ ]:
def get_first_token_attn(model, loader, n_batches=SINK_BATCHES):
    """Per-layer mean attention to token-0 (attention-sink proxy)."""
    n_layers = model.config.n_layer
    n_heads  = model.config.n_head
    head_dim = model.config.n_embd // n_heads
    all_first = [[] for _ in range(n_layers)]
    handles   = []

    for li, block in enumerate(model.transformer.h):
        cap = {}
        def make_hook(idx, c):
            def hook(module, inp, out):
                hs = inp[0]; B, N, _ = hs.shape
                qkv = module.c_attn(hs)
                q, k, _ = qkv.split(module.embed_dim, dim=2)
                q = q.view(B, N, n_heads, head_dim).permute(0,2,1,3)
                k = k.view(B, N, n_heads, head_dim).permute(0,2,1,3)
                scores = (q * head_dim**-0.5) @ k.transpose(-2,-1)
                cm = torch.triu(torch.ones(N, N, device=hs.device, dtype=torch.bool), diagonal=1)
                scores = scores.masked_fill(cm[None,None], float('-inf'))
                w = scores.softmax(-1)
                c['first'] = w[:,:,:,0].mean(dim=(1,2)).detach()
            return hook
        h = block.attn.register_forward_hook(make_hook(li, cap))
        handles.append((li, cap, h))

    model.eval()
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= n_batches: break
            model(batch.to(DEVICE))
            for li, cap, _ in handles:
                if 'first' in cap:
                    all_first[li].append(cap['first'].mean().item())

    for _, _, h in handles: h.remove()
    return [np.mean(v) if v else 0.0 for v in all_first]


sink_scores = {}
for name, pos in EXPERIMENTS:
    print(f'Attention sink: {name} ...')
    m = load_model(name, pos)
    sink_scores[name] = get_first_token_attn(m, val_loader)
    del m
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    print(f'  mean = {np.mean(sink_scores[name]):.4f}')

## 5. Gate Statistics (G1, G2, G3, G4, G5, PNG)
Collects per-layer gate scores and gate-vs-norm correlations for all gated variants.

In [ ]:
@torch.no_grad()
def collect_gate_scores(model, loader, n_batches=GATE_BATCHES):
    """Mean gate activation per layer."""
    for block in model.transformer.h:
        block.attn._capture_gate = True
    all_gates = [[] for _ in range(model.config.n_layer)]
    model.eval()
    for i, batch in enumerate(loader):
        if i >= n_batches: break
        model(batch.to(DEVICE))
        for li, block in enumerate(model.transformer.h):
            if hasattr(block.attn, '_last_gate'):
                all_gates[li].append(block.attn._last_gate.mean().item())
    for block in model.transformer.h:
        block.attn._capture_gate = False
    return [np.mean(v) if v else 0.0 for v in all_gates]


@torch.no_grad()
def collect_gate_head_matrix(model, loader, n_batches=GATE_BATCHES):
    """Layer × Head mean gate matrix."""
    n_layers = model.config.n_layer; n_heads = model.config.n_head
    for block in model.transformer.h:
        block.attn._capture_gate = True
    lh = np.zeros((n_layers, n_heads)); nc = 0
    model.eval()
    for i, batch in enumerate(loader):
        if i >= n_batches: break
        model(batch.to(DEVICE))
        for li, block in enumerate(model.transformer.h):
            if hasattr(block.attn, '_last_gate'):
                lh[li] += block.attn._last_gate.mean(dim=(0,1)).cpu().numpy()
        nc += 1
    for block in model.transformer.h:
        block.attn._capture_gate = False
    return lh / max(nc, 1)


@torch.no_grad()
def collect_gate_vs_norm(model, loader, n_batches=GATE_BATCHES):
    """Gate scores vs token norms from the last layer."""
    last = model.transformer.h[-1]; last.attn._capture_gate = True
    all_norms, all_gates = [], []
    model.eval()
    for i, batch in enumerate(loader):
        if i >= n_batches: break
        model(batch.to(DEVICE))
        if hasattr(last.attn, '_last_gate'):
            all_gates.append(last.attn._last_gate.mean(-1).cpu().flatten().numpy())
            all_norms.append(last.attn._last_x_norm.cpu().flatten().numpy())
    last.attn._capture_gate = False
    if not all_gates: return np.array([]), np.array([])
    return np.concatenate(all_norms), np.concatenate(all_gates)


gate_layer_scores = {}   # name → list[float] per layer
gate_head_matrix  = {}   # name → (n_layers, n_heads) array
gate_norm_data    = {}   # name → (norms, gates) arrays
pearson_r         = {}   # name → (r, p)

GATED_VARIANTS = [(n, p) for n, p in EXPERIMENTS if p != 'baseline']

for name, pos in GATED_VARIANTS:
    print(f'Gate stats: {name} ...')
    m = load_model(name, pos)
    gate_layer_scores[name] = collect_gate_scores(m, val_loader)
    gate_head_matrix[name]  = collect_gate_head_matrix(m, val_loader)
    norms, gates            = collect_gate_vs_norm(m, val_loader)
    gate_norm_data[name]    = (norms, gates)
    del m
    if DEVICE == 'cuda': torch.cuda.empty_cache()

    if len(norms) > 1:
        rng = np.random.default_rng(42)
        idx = rng.choice(len(norms), min(5000, len(norms)), replace=False)
        r, p = pearsonr(norms[idx], gates[idx])
        pearson_r[name] = (r, p)
        print(f'  per-layer: {[f"{s:.3f}" for s in gate_layer_scores[name]]}')
        print(f'  Pearson r={r:.3f}  p={p:.2e}')
    else:
        pearson_r[name] = (float('nan'), float('nan'))

## 6. Word-Level Attention Matrices

In [ ]:
# Select a readable sample from the validation set
for ci in range(len(val_ids)):
    text = tokenizer.decode(val_ids[ci].tolist(), skip_special_tokens=True)
    if len(text.strip()) > 80 and '=' not in text[:30]:
        SAMPLE_IDX = ci
        break

SAMPLE_IDS = val_ids[SAMPLE_IDX].unsqueeze(0)
sample_tokens  = tokenizer.convert_ids_to_tokens(SAMPLE_IDS[0, :DISPLAY_LEN].tolist())
display_tokens = [t.replace('\u0120', ' ').replace('\u010a', '\\n') for t in sample_tokens]
print(f'Sample [{SAMPLE_IDX}]: {" ".join(display_tokens[:20])} ...')


def _attn_via_hook(model, input_ids, n_tokens):
    """Manual softmax attention via forward hook — works on any GPT-2 model."""
    n_heads = model.config.n_head; head_dim = model.config.n_embd // n_heads
    attn_layers, handles = [], []
    for block in model.transformer.h:
        cap = {}
        def make_hook(c):
            def hook(module, inp, out):
                hs = inp[0]; B, N, _ = hs.shape
                qkv = module.c_attn(hs)
                q, k, _ = qkv.split(module.embed_dim, dim=2)
                q = q.view(B,N,n_heads,head_dim).permute(0,2,1,3)
                k = k.view(B,N,n_heads,head_dim).permute(0,2,1,3)
                scores = (q * head_dim**-0.5) @ k.transpose(-2,-1)
                cm = torch.triu(torch.ones(N,N,device=hs.device,dtype=torch.bool), diagonal=1)
                scores = scores.masked_fill(cm[None,None], float('-inf'))
                c['attn'] = scores.softmax(-1)[0].detach().cpu().numpy()
            return hook
        h = block.attn.register_forward_hook(make_hook(cap))
        handles.append((cap, h))
    model.eval()
    with torch.no_grad():
        model(input_ids.to(DEVICE))
    for cap, h in handles:
        h.remove()
        if 'attn' in cap: attn_layers.append(cap['attn'])
    if not attn_layers: return np.zeros((n_tokens, n_tokens))
    return np.stack(attn_layers).mean(axis=(0,1))[:n_tokens, :n_tokens]


def get_attn_matrix(model, input_ids, n_tokens=DISPLAY_LEN, is_baseline=False):
    model.eval()
    if is_baseline:
        _orig = getattr(model.config, '_attn_implementation', 'sdpa')
        model.config._attn_implementation = 'eager'
        with torch.no_grad():
            out = model(input_ids.to(DEVICE), output_attentions=True)
        model.config._attn_implementation = _orig
        if out.attentions is None:
            return _attn_via_hook(model, input_ids, n_tokens)
        attn_layers = [a[0].cpu().numpy() for a in out.attentions]
    else:
        for block in model.transformer.h:
            block.attn._capture_attn = True
        with torch.no_grad():
            model(input_ids.to(DEVICE))
        attn_layers = []
        for block in model.transformer.h:
            if hasattr(block.attn, '_last_attn'):
                attn_layers.append(block.attn._last_attn[0].cpu().numpy())
            block.attn._capture_attn = False
    if not attn_layers:
        return _attn_via_hook(model, input_ids, n_tokens)
    return np.stack(attn_layers).mean(axis=(0,1))[:n_tokens, :n_tokens]


attn_matrices = {}
for name, pos in EXPERIMENTS:
    print(f'Attention matrix: {name} ...')
    m = load_model(name, pos)
    attn_matrices[name] = get_attn_matrix(m, SAMPLE_IDS, is_baseline=(pos=='baseline'))
    del m
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    print(f'  max={attn_matrices[name].max():.4f}')

## 7. Figures

In [ ]:
sns.set_theme(style='whitegrid', font_scale=1.1)

names      = [n for n, _ in EXPERIMENTS]
test_ppl_v = [test_ppls[n] for n in names]
val_ppl_v  = [val_ppls[n]  for n in names]
base_ppl   = test_ppls['Baseline']
deltas     = [base_ppl - p for p in test_ppl_v]
colors     = [PALETTE[n] for n in names]

def savefig(name):
    path = FIG_DIR / name
    plt.savefig(path, bbox_inches='tight', dpi=150)
    print(f'  Saved {path}')

In [ ]:
# ── Fig 1 — Test PPL bar chart + delta ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Fig 1  |  Test Perplexity — All Variants\n'
             'GPT-2 small · WikiText-103 · Unfrozen backbone',
             fontsize=14, fontweight='bold')

ax = axes[0]
bars = ax.bar(names, test_ppl_v, color=colors, edgecolor='k', linewidth=0.6)
for bar, v in zip(bars, test_ppl_v):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            f'{v:.2f}', ha='center', va='bottom', fontsize=8)
ax.axhline(base_ppl, color='gray', linestyle='--', lw=1.2, label='Baseline')
ax.set_ylabel('Test Perplexity (↓ better)')
ax.set_title('Absolute Test PPL', fontweight='bold')
ax.set_xticklabels(names, rotation=30, ha='right')
ax.set_ylim(min(test_ppl_v)*0.97, max(test_ppl_v)*1.01)
ax.grid(axis='y', alpha=0.3); ax.legend()

ax2 = axes[1]
dcol = ['#2ca02c' if d > 0 else '#d62728' for d in deltas]
bars2 = ax2.bar(names, deltas, color=dcol, edgecolor='k', linewidth=0.6)
for bar, d in zip(bars2, deltas):
    ax2.text(bar.get_x()+bar.get_width()/2,
             bar.get_height() + (0.05 if d >= 0 else -0.3),
             f'{d:+.2f}', ha='center', va='bottom', fontsize=8)
ax2.axhline(0, color='k', lw=1)
ax2.set_ylabel('ΔPPL vs Baseline (↑ = improvement)')
ax2.set_title('PPL Delta vs Baseline', fontweight='bold')
ax2.set_xticklabels(names, rotation=30, ha='right')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout(); savefig('fig1_ppl_bar.pdf'); plt.show()

In [ ]:
# ── Fig 2 — Val PPL vs Test PPL scatter ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
for name in names:
    ax.scatter(val_ppls[name], test_ppls[name], color=PALETTE[name], s=120, zorder=5,
               edgecolors='k', linewidths=0.7)
    ax.annotate(name, (val_ppls[name], test_ppls[name]),
                textcoords='offset points', xytext=(7, 0), fontsize=8)

# diagonal reference
all_ppl = list(val_ppls.values()) + list(test_ppls.values())
lim = (min(all_ppl)*0.99, max(all_ppl)*1.01)
ax.plot(lim, lim, 'k--', lw=1, alpha=0.4, label='val = test')
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel('Validation PPL', fontsize=12)
ax.set_ylabel('Test PPL', fontsize=12)
ax.set_title('Fig 2  |  Val PPL vs Test PPL\nConsistency across splits', fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(alpha=0.3)

plt.tight_layout(); savefig('fig2_val_vs_test_ppl.pdf'); plt.show()

In [ ]:
# ── Fig 3 — Training curves (loss + val PPL) ────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Fig 3  |  Training Dynamics — All Variants\nGPT-2 small · WikiText-103',
             fontsize=14, fontweight='bold')

for name, _ in EXPERIMENTS:
    log = load_log(name)
    if not log['step']: continue
    lw = 2.5 if name == 'PNG — Norm-Aware' else 1.2
    ls = '-'  if name in ('Baseline', 'PNG — Norm-Aware') else '--'
    ax1.plot(log['step'], log['train_loss'], color=PALETTE[name], label=name, lw=lw, ls=ls, alpha=0.9)
    ax2.plot(log['step'], log['val_ppl'],    color=PALETTE[name], label=name, lw=lw, ls=ls, alpha=0.9)

ax1.set_xlabel('Step'); ax1.set_ylabel('Train Loss')
ax1.set_title('Training Loss', fontweight='bold')
ax1.legend(fontsize=7); ax1.grid(alpha=0.3)

ax2.set_xlabel('Step'); ax2.set_ylabel('Val PPL')
ax2.set_title('Validation Perplexity', fontweight='bold')
ax2.legend(fontsize=7); ax2.grid(alpha=0.3)

plt.tight_layout(); savefig('fig3_training_curves.pdf'); plt.show()

In [ ]:
# ── Fig 4 — Attention Sink — all variants ───────────────────────────────────
n_layers = 12
layers_x = list(range(1, n_layers + 1))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Fig 4  |  Attention Sink — First-Token Attention Score\n'
             'All variants · GPT-2 small · WikiText-103 val',
             fontsize=14, fontweight='bold')

markers = {'Baseline':'o','G1 — SDPA output':'s','G2 — Value proj':'^',
           'G3 — Key proj':'D','G4 — Query proj':'v','G5 — Dense output':'<','PNG — Norm-Aware':'*'}
for name, _ in EXPERIMENTS:
    vals = sink_scores[name]
    lw = 2.5 if name in ('Baseline', 'PNG — Norm-Aware') else 1.2
    ax1.plot(layers_x, vals, marker=markers.get(name,'o'), linestyle='-',
             color=PALETTE[name], label=name, lw=lw, markersize=5)

ax1.set_xlabel('Layer'); ax1.set_ylabel('Mean Attention to First Token')
ax1.set_title('Per-Layer Attention Sink', fontweight='bold')
ax1.legend(fontsize=7); ax1.grid(alpha=0.3)

means = [np.mean(sink_scores[n]) for n in names]
bars  = ax2.bar(names, means, color=colors, edgecolor='k', linewidth=0.6)
for bar, v in zip(bars, means):
    ax2.text(bar.get_x()+bar.get_width()/2, v+0.0005,
             f'{v:.4f}', ha='center', va='bottom', fontsize=7, fontweight='bold')
ax2.set_ylabel('Mean First-Token Attention (all layers)')
ax2.set_title('Overall Attention Sink', fontweight='bold')
ax2.set_xticklabels(names, rotation=30, ha='right')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout(); savefig('fig4_attention_sink.pdf'); plt.show()

# Print reduction vs baseline
base_sink = np.mean(sink_scores['Baseline'])
print('\nAttention sink reduction vs Baseline:')
for name in names:
    m = np.mean(sink_scores[name])
    red = (1 - m / base_sink) * 100
    print(f'  {name:<26}  mean={m:.4f}  reduction={red:+.1f}%')

In [ ]:
# ── Fig 5 — Gate Sparsity per Layer — all gated variants ────────────────────
n_gated = len(GATED_VARIANTS)
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
fig.suptitle('Fig 5  |  Gate Sparsity per Layer\n'
             'Mean gate score per layer — scores < 0.5 → active suppression',
             fontsize=14, fontweight='bold')
axes = axes.flatten()

for i, (name, pos) in enumerate(GATED_VARIANTS):
    ax = axes[i]
    scores = gate_layer_scores.get(name, [])
    ax.bar(range(1, len(scores)+1), scores, color=PALETTE[name], edgecolor='k', linewidth=0.4)
    ax.axhline(0.5, color='red', linestyle='--', lw=1.2, label='neutral (σ=0.5)')
    ax.set_xlabel('Layer'); ax.set_ylabel('Mean Gate Score')
    ax.set_title(name, fontweight='bold', color=PALETTE[name])
    ax.set_ylim(0, 1); ax.legend(fontsize=7); ax.grid(axis='y', alpha=0.3)

# Hide unused subplots
for j in range(n_gated, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout(); savefig('fig5_gate_sparsity_all.pdf'); plt.show()

In [ ]:
# ── Fig 6 — Layer × Head Gate Activation Heatmaps ───────────────────────────
n_gated = len(GATED_VARIANTS)
fig, axes = plt.subplots(2, 3, figsize=(22, 12))
fig.suptitle('Fig 6  |  Gate Activations — Layer × Head\n'
             'Mean over batch & tokens — all gated variants',
             fontsize=14, fontweight='bold')
axes = axes.flatten()

n_layers = 12; n_heads = 12

for i, (name, pos) in enumerate(GATED_VARIANTS):
    ax = axes[i]
    lh = gate_head_matrix.get(name, np.zeros((n_layers, n_heads)))
    sns.heatmap(lh, annot=True, fmt='.2f', cmap='RdYlGn',
                xticklabels=[f'H{j+1}' for j in range(n_heads)],
                yticklabels=[f'L{j+1}' for j in range(n_layers)],
                vmin=0, vmax=1, ax=ax, linewidths=0.3,
                annot_kws={'size': 6}, cbar_kws={'shrink': 0.8})
    ax.set_xlabel('Head', fontsize=9); ax.set_ylabel('Layer', fontsize=9)
    ax.set_title(name, fontweight='bold', fontsize=11)

for j in range(n_gated, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout(); savefig('fig6_gate_head_heatmaps.pdf'); plt.show()

In [ ]:
# ── Fig 7 — Gate Score vs Token Norm (last layer) ───────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Fig 7  |  Gate Score vs Token L2 Norm — Last Layer\n'
             'PNG explicitly conditions on norm → expect strongest correlation',
             fontsize=14, fontweight='bold')
axes = axes.flatten()

for i, (name, pos) in enumerate(GATED_VARIANTS):
    ax = axes[i]
    norms, gates = gate_norm_data.get(name, (np.array([]), np.array([])))
    r, p = pearson_r.get(name, (float('nan'), float('nan')))
    if len(norms) > 0:
        rng = np.random.default_rng(42)
        idx = rng.choice(len(norms), min(3000, len(norms)), replace=False)
        ax.scatter(norms[idx], gates[idx], alpha=0.15, s=5,
                   color=PALETTE[name], rasterized=True)
        z  = np.polyfit(norms[idx], gates[idx], 1)
        xr = np.linspace(norms[idx].min(), norms[idx].max(), 200)
        ax.plot(xr, np.polyval(z, xr), 'r-', lw=2,
                label=f'r={r:.3f}  p={p:.1e}')
        ax.legend(fontsize=8)
    ax.set_xlabel('||x||₂', fontsize=10)
    ax.set_ylabel('Gate score σ(·)', fontsize=10)
    ax.set_title(name, fontweight='bold', color=PALETTE[name])
    ax.grid(alpha=0.3)

for j in range(n_gated, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout(); savefig('fig7_gate_vs_norm.pdf'); plt.show()

In [ ]:
# ── Fig 8 — Gate-Norm Pearson r summary ─────────────────────────────────────
r_vals = [pearson_r.get(n, (float('nan'),))[0] for n, p in EXPERIMENTS if p != 'baseline']
r_names = [n for n, p in EXPERIMENTS if p != 'baseline']
r_colors = [PALETTE[n] for n in r_names]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(r_names, r_vals, color=r_colors, edgecolor='k', linewidth=0.6)
for bar, v in zip(bars, r_vals):
    if not np.isnan(v):
        ax.text(bar.get_x()+bar.get_width()/2, v + 0.01,
                f'{v:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.axhline(0, color='k', lw=1)
ax.set_ylabel('Pearson r  (gate score vs token norm)', fontsize=11)
ax.set_title('Fig 8  |  Gate-Norm Correlation Summary\n'
             'PNG expected to show highest r (norm explicitly in gate equation)',
             fontsize=13, fontweight='bold')
ax.set_xticklabels(r_names, rotation=25, ha='right')
ax.set_ylim(-0.1, 1.0); ax.grid(axis='y', alpha=0.3)

plt.tight_layout(); savefig('fig8_pearson_r_summary.pdf'); plt.show()

In [ ]:
# ── Fig 9 — Word-Level Attention Heatmaps — All 7 Variants ─────────────────
fig, axes = plt.subplots(2, 4, figsize=(28, 14))
fig.suptitle('Fig 9  |  Word-Level Attention Heatmap — All Variants\n'
             'Mean over all layers & heads · GPT-2 small · WikiText-103 val',
             fontsize=14, fontweight='bold')
axes = axes.flatten()

for i, (name, pos) in enumerate(EXPERIMENTS):
    ax = axes[i]
    mat  = attn_matrices[name]
    mask = np.triu(np.ones_like(mat, dtype=bool), k=1)
    sns.heatmap(mat, ax=ax, mask=mask, cmap='YlOrRd',
                xticklabels=display_tokens, yticklabels=display_tokens,
                cbar=True, linewidths=0.0, vmin=0)
    ax.set_title(name, fontsize=10, fontweight='bold')
    ax.set_xlabel('Key (attended to)', fontsize=7)
    ax.set_ylabel('Query (attending from)', fontsize=7)
    ax.set_xticklabels(display_tokens, rotation=90, fontsize=5)
    ax.set_yticklabels(display_tokens, rotation=0,  fontsize=5)

# Hide the 8th subplot (7 variants only)
axes[7].set_visible(False)

plt.tight_layout(); savefig('fig9_word_attention_all.pdf'); plt.show()

In [ ]:
# ── Fig 10 — Per-Token Attention Received — All 7 Variants ─────────────────
fig, axes = plt.subplots(7, 1, figsize=(20, 21))
fig.suptitle('Fig 10  |  Per-Token Attention Received (Column Sum)\n'
             'How much each token is attended to — averaged over all layers & heads',
             fontsize=14, fontweight='bold')

for ax, (name, pos) in zip(axes, EXPERIMENTS):
    mat     = attn_matrices[name]
    col_sum = mat.sum(axis=0); col_sum = col_sum / col_sum.sum()
    bar_colors = plt.cm.YlOrRd(col_sum / col_sum.max())
    ax.bar(range(len(display_tokens)), col_sum, color=bar_colors, edgecolor='none')
    ax.set_xticks(range(len(display_tokens)))
    ax.set_xticklabels(display_tokens, rotation=70, ha='right', fontsize=6)
    ax.set_ylabel('Attn weight', fontsize=8)
    ax.set_title(name, fontweight='bold', fontsize=9, color=PALETTE[name])
    ax.grid(axis='y', alpha=0.3)
    top3 = np.argsort(col_sum)[-3:][::-1]
    for t in top3:
        ax.text(t, col_sum[t]+0.002, '▲', ha='center', va='bottom', fontsize=8, color='red')

plt.tight_layout(); savefig('fig10_per_token_attn_all.pdf'); plt.show()

In [ ]:
# ── Fig 11 — Attention Sink Heatmap: Variant × Layer ────────────────────────
sink_matrix = np.array([sink_scores[n] for n in names])  # (7, 12)

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(sink_matrix, annot=True, fmt='.3f', cmap='YlOrRd',
            xticklabels=[f'L{i+1}' for i in range(12)],
            yticklabels=names,
            ax=ax, linewidths=0.4, cbar_kws={'label': 'Mean attn to first token'})
ax.set_xlabel('Layer', fontsize=12)
ax.set_title('Fig 11  |  Attention Sink Heatmap — Variant × Layer\n'
             'Darker = more attention concentrated on first token',
             fontsize=13, fontweight='bold')

plt.tight_layout(); savefig('fig11_sink_heatmap.pdf'); plt.show()

In [ ]:
# ── Fig 12 — Radar chart: multi-metric comparison ───────────────────────────
# Metrics: test PPL (inverted), val PPL (inverted), attention sink reduction,
#          mean gate score deviation from 0.5, Pearson r (gate-norm)

def norm01(vals, invert=False):
    a, b = min(vals), max(vals)
    if b == a: return [0.5] * len(vals)
    out = [(v - a) / (b - a) for v in vals]
    return [1 - v for v in out] if invert else out

base_sink = np.mean(sink_scores['Baseline'])
metric_data = {}
for name, _ in EXPERIMENTS:
    sink_red  = max(0, (base_sink - np.mean(sink_scores[name])) / base_sink)
    gate_act  = np.mean([abs(s - 0.5) for s in gate_layer_scores.get(name, [0.5]*12)])
    r_val     = pearson_r.get(name, (0,))[0]
    metric_data[name] = {
        'test_ppl':    test_ppls[name],
        'val_ppl':     val_ppls[name],
        'sink_red':    sink_red,
        'gate_act':    gate_act,
        'pearson_r':   r_val if not np.isnan(r_val) else 0.0,
    }

metrics    = ['Test PPL\n(lower=better)', 'Val PPL\n(lower=better)',
              'Sink\nReduction', 'Gate\nActivation', 'Gate-Norm\nCorr (r)']
metric_keys = ['test_ppl', 'val_ppl', 'sink_red', 'gate_act', 'pearson_r']
invert      = [True, True, False, False, False]

norm_vals = {}
for j, (key, inv) in enumerate(zip(metric_keys, invert)):
    raw = [metric_data[n][key] for n in names]
    nrm = norm01(raw, invert=inv)
    for i, name in enumerate(names):
        norm_vals.setdefault(name, []).append(nrm[i])

N = len(metrics)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
for name in names:
    vals = norm_vals[name] + norm_vals[name][:1]
    ax.plot(angles, vals, color=PALETTE[name], linewidth=2,
            label=name, marker='o', markersize=5)
    ax.fill(angles, vals, color=PALETTE[name], alpha=0.07)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics, fontsize=10)
ax.set_yticklabels([]); ax.set_ylim(0, 1)
ax.set_title('Fig 12  |  Multi-Metric Radar Chart\n'
             'Normalised scores — larger = better on each axis',
             fontsize=13, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=9)

plt.tight_layout(); savefig('fig12_radar_chart.pdf'); plt.show()

## 8. Results Summary

In [ ]:
print('=' * 75)
print('  EVALUATION SUMMARY — GPT-2 small · WikiText-103 · Unfrozen backbone')
print('=' * 75)
print(f'  {"Variant":<26}  {"Val PPL":>8}  {"Test PPL":>9}  {"ΔPPL":>7}  {"Sink":>6}  {"r(G,N)":>7}')
print('  ' + '-'*70)

best_test = min(test_ppls.values())
for name, _ in EXPERIMENTS:
    tp    = test_ppls[name]
    vp    = val_ppls[name]
    delta = base_ppl - tp
    sink  = np.mean(sink_scores[name])
    r     = pearson_r.get(name, (float('nan'),))[0]
    r_s   = f'{r:.3f}' if not np.isnan(r) else '  —'
    best  = ' ←' if tp == best_test else ''
    print(f'  {name:<26}  {vp:>8.2f}  {tp:>9.2f}  {delta:>+7.2f}  {sink:>6.4f}  {r_s:>7}{best}')

print()
print(f'  Baseline test PPL        : {base_ppl:.2f}')
print(f'  Best test PPL            : {best_test:.2f}  ({[n for n,_ in EXPERIMENTS if test_ppls[n]==best_test][0]})')
print(f'  Lowest attention sink    : {min(np.mean(sink_scores[n]) for n in names):.4f}')
print()
print('  Figures saved to:', FIG_DIR.resolve())
for f in sorted(FIG_DIR.iterdir()):
    print(f'    {f.name}')
print('=' * 75)